# محرك عثمان: توقيت مصحف كامل · Othman Engine: time a whole mushaf

يوقّت كل سورة من روابط الصوت، ويحفظ ملف JSON لكل سورة في Google Drive أولاً بأول.
Times every surah from audio URLs and saves one JSON per surah to Google Drive as it goes.

**قبل البدء · Before you start**
1. اقبل شروط النموذج من [صفحته](https://huggingface.co/Muno459/fastconformer-quran)، وأنشئ [token](https://huggingface.co/settings/tokens) (صلاحية Read).
2. من أيقونة المفتاح 🔑 يسار الصفحة أضف سراً باسم `HF_TOKEN` (و`GH_TOKEN` ما دام المستودع خاصاً)، وفعّل **Notebook access**.
3. يكفي وقت التشغيل العادي بالمعالج (CPU)؛ المحرك لا يستخدم كرت الشاشة. · A CPU runtime is enough; the engine does not use a GPU.

**إذا انقطعت الجلسة · If the session drops:** شغّل كل الخلايا من جديد (Runtime ← Run all).
السور المحفوظة في Drive تُتخطى، ويكمل من أول سورة لم تكتمل. السورة التي فشلت تُعاد عند التشغيل التالي.
Run all cells again: surahs already in Drive are skipped and it resumes from the first unfinished one.

## الإعدادات · Settings
- `SOURCES`: سطر لكل رابط. **رابط مجلد** ينتهي بـ `/` تُضاف له السور من `SURAHS` (`001.mp3` ...)،
  أو **رابط ملف** واحد ورقم السورة من اسمه (`002.mp3`).
- **مصحف عندك:** ارفع مجلده إلى Google Drive واكتب مساره، مثل `/content/drive/MyDrive/مصاحف/عبدالعزيز التركي`.
  السورة تُعرف من اسم كل ملف: رقم (`002.mp3`) أو «سورة عبس»، واسم المجلد هو اسم القارئ في النتائج.
- القيم الحالية هي عينة المقارنة (3 قراء × 5 سور). للمصحف كاملاً اترك رابط قارئ واحد واكتب `SURAHS = "1-114"`.
- الناتج: `OUT_DIR/<المحرك>/<القارئ>/002.json`، يُحفظ بعد كل سورة.

One line per URL: a **folder** ending in `/` (surahs from `SURAHS` are added) or a single **file** (surah from its name),
or a Google Drive folder path (the surah is read from each file name: a number or «سورة <name>»).
The defaults are the comparison sample. Results are saved to Drive after every surah.

In [ ]:
SOURCES = """
https://cdn.mp3quran.net/audio/abdulrahman-sudais/r1/
https://cdn.mp3quran.net/audio/mishary-alafasy/r1/
https://cdn.mp3quran.net/audio/maher-muaiqly/r3/
"""
SURAHS = "2,18,67,78,97"  # للمصحف كاملاً · whole mushaf: "1-114"
OUT_DIR = "/content/drive/MyDrive/othman-timings"

## 1. التجهيز · Setup
يربط Drive ويثبّت المحرك ويحمّل النموذج (~1GB، نحو دقيقة) · Mounts Drive, installs the engine and downloads the model.

In [ ]:
import os
import sys
from google.colab import drive, userdata


def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None


drive.mount("/content/drive")
BRANCH = "main"
# GH_TOKEN مطلوب فقط ما دام المستودع خاصاً · only needed while the repository is private
gh = secret("GH_TOKEN")
repo = f"https://{gh}@github.com/vvd7mvv/othman-engine" if gh else "https://github.com/vvd7mvv/othman-engine"
!rm -rf /content/othman-engine && git clone -q --depth 1 -b {BRANCH} {repo} /content/othman-engine
sys.path.insert(0, "/content/othman-engine/notebooks")
sys.modules.pop("timings", None)  # عند إعادة التشغيل في نفس الجلسة: نحمّل النسخة الجديدة لا القديمة من الذاكرة
import timings

os.environ["HF_TOKEN"] = secret("HF_TOKEN") or ""
if not os.environ["HF_TOKEN"]:
    print("⚠️ HF_TOKEN غير موجود: أضفه من 🔑 وفعّل Notebook access · HF_TOKEN missing")
!pip install -q /content/othman-engine
!othman-engine download
if not os.path.exists(os.path.expanduser("~/.othman-engine/models/fastconformer-quran/onnx/model.onnx")):
    raise RuntimeError("لم يكتمل تحميل النموذج؛ اقرأ الرسالة أعلاه · The model was not downloaded; see above")

## 2. التوقيت · Align
قبل أي سورة أطول من نصف ساعة (مثل البقرة) يكتب الوقت المتوقع. على معالج Colab المجاني (نواتان) البقرة نحو 10–15 دقيقة.
Before any surah longer than 30 minutes it prints the expected time (Al-Baqarah takes about 10–15 min on the free CPU).

In [ ]:
import platform
from importlib.metadata import version

from othman_engine import Engine

engine = Engine()


def align(path, surah):
    ayahs = {}
    for segment in engine.align_file(path).quran_segments():
        # الآية المكررة: نحفظ أول قراءة. وآيات سورة أخرى (خطأ تعرّف) لا تدخل.
        if segment.surah == surah and segment.ayah not in ayahs:
            ayahs[segment.ayah] = (segment.start, segment.end, segment.confidence)
    return ayahs


cpu = next((l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")), platform.processor())
device = f"CPU {os.cpu_count()} threads ({cpu})"
print(device)
failures = timings.run_batch(timings.parse_sources(SOURCES, SURAHS), OUT_DIR, "othman-engine", align, device,
                             version=version("othman-engine"), speed_guess=0.13)